# <span style="color:#6495D6">Bodemvocht Animaties voor Waterschappen</span>
Dit notebook maakt animaties van bodemvochtgegevens afkomstig uit WIWB voor een geselecteerd waterschap.

---

#### <span style="color:#C78152"><b>Functionaliteit</b></span>
* Haalt historische bodemvocht GeoTIFFs op via de WIWB API.
* Knipt rasters uit op basis van een shapefile van het geselecteerde waterschap.
* Slaat de bodemvocht GeoTIFFs lokaal op.
* Maakt een animatie van de geselecteerde periode voor het geselecteerde waterschap.


#### <span style="color:#C78152"><b>Gebruiksaanwijzing</b></span>
* Vul het pad naar deze notebook en uw WIWB credentials in 
* Kies een waterschap uit de lijst van beschikbare waterschappen.
* Controleer of de shapefile van het waterschap aanwezig is in de opgegeven map.
* Voeg de WIWB credentials toe en voer de analyses uit

#### <span style="color:#6495D6"><b>----</b></span>
<span style="color:#6495D6">HydroLogic, Amersfoort</span><br>
<span style="color:#6495D6">Created on Tue Sept 11 2025</span>

In [ ]:
# -*- coding: utf-8 -*-

# importeer packages
import os
from pathlib import Path
import numpy as np
import geopandas as gpd
import rasterio, datetime, imageio
import requests, jwt, base64, warnings
from rasterio.mask import mask
from typing import Dict
from dotenv import load_dotenv

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
from matplotlib.gridspec import GridSpec
from IPython.display import HTML, display, clear_output



# initialiseer verwijzing naar ffmpeg.exe voor animatie
mpl.rcParams['animation.ffmpeg_path'] = imageio.plugins.ffmpeg.get_exe()


# =======================
#        SETTINGS   
# =======================

# kies een waterschap
SELECTED_WATERSCHAP = "Waterschap Limburg"

# lijst van beschikbare waterschappen:
#   •	De Stichtse Rijnlanden
#   •	HH Hollands Noorderkwartier
#   •	HHS van Delfland
#   •	Aa en Maas
#   •	Brabantse Delta
#   •	Waterschap De Dommel
#   •	Waterschap Hollandse Delta
#   •	HH van Rijnland
#   •	Schieland en de Krimpenerwaard
#   •	HH Amstel, Gooi en Vecht
#   •	Waterschap Hunze en Aa's
#   •	Waterschap Noorderzijlvest
#   •	Waterschap Limburg
#   •	Waterschap Drents Overijsselse Delta
#   •	Waterschap Rijn en IJssel
#   •	Waterschap Rivierenland
#   •	Waterschap Scheldestromen
#   •	Vechtstromen
#   •	Vallei & Veluwe
#   •	Waterschap Zuiderzeeland
#   •	Wetterskip Fryslân


# selecteer een periode
start = datetime.date(2025, 1, 1)
end = datetime.date(2025, 1, 30)


DIR_BASE = Path(r"...")

# locatie naar map met shapefile
DIR_SHAPE = Path(os.path.join(DIR_BASE, r"static\shape\waterschapsgrenzen.shp"))
# locatie naar map met bodemvocht bestanden
DIR_INPUT = Path(os.path.join(DIR_BASE, r"static\input"))
# locatie naar map waar animatie opgeslagen moet worden
DIR_OUTPUT = Path(os.path.join(DIR_BASE, r"static\output"))

# interval tussen frames [ms]
FRAME_INTERVAL = 500
# titel van de plot
PLOT_TITLE = SELECTED_WATERSCHAP
# minimum en maximum van de kleurschaal
VMIN, VMAX = 0.4, 0.5
# type kleurschaal (bijv. "viridis", "plasma", "YlGnBu")
CMAP = "YlGnBu"

# load variables from .env into environment
load_dotenv(DIR_BASE / "secrets/.env")
# WIWB api url
API_URL = "https://wiwb.hydronet.com/api/grids/get"
# WIWB client id
CLIENT_ID = os.getenv("CLIENT_ID_WIWB")
# WIWB client secret
CLIENT_SECRET = os.getenv("CLIENT_SECRET_WIWB")

if CLIENT_ID is None or CLIENT_SECRET is None:
    raise ValueError("CLIENT_ID or CLIENT_SECRET not found in environment variables")

#### <span style="color:#6495D6"><b>Lees shapefile en selecteer het waterschap</b></span>

In deze stap wordt de shapefile van alle waterschapsgrenzen ingelezen en de geometrie van het geselecteerde waterschap opgehaald.

##### <span style="color:#C78152"><b>Functionaliteit:</b></span>

* Lees de shapefile en zet het coordinatensysteem om zodat deze overeenkomt met de bodemvocht rasterdata.
* Selecteer de geometrie van het gekozen waterschap (`SELECTED_WATERSCHAP`).
* Controleer of het gekozen waterschap in de shapefile aanwezig is.
* Sla de grenzen van het waterschap op in variabelen (`xmin`, `ymin`, `xmax`, `ymax`) en in de `geometry` variabele voor verdere analyses.


In [ ]:
# -*- coding: utf-8 -*-

# lees shapefile en zet CRS om naar het CRS van de rasterbestanden
gdf = gpd.read_file(DIR_SHAPE).to_crs(epsg=28992)
# selecteer geometrie van het waterschap
ws_geom = gdf.loc[gdf["waterschap"] == SELECTED_WATERSCHAP, "geometry"]

if ws_geom.empty:
    available_ws = "\n - " + "\n - ".join(list(gdf["waterschap"]))
    print(
        f"[ERROR] Waterschap '{SELECTED_WATERSCHAP}' niet gevonden in shapefile. De beschikbare waterschappen zijn: {available_ws}"
    )
else:
    xmin, ymin, xmax, ymax  = ws_geom.values[0].bounds
    # selecteer geometie van het waterschap
    geometry = [ws_geom.values[0]]

    # bereken lengte en breedte van figuur
    width = xmax - xmin
    height = ymax - ymin

    # bereken figuur hoogte obv ratio
    fig_height = max(6, min(6 * (height / width), 12))
    # herbereken width wanneer er een cap wordt gezet
    fig_width = max(6, min(fig_height * (width / height), 12))
    

#### <span style="color:#6495D6"><b>Authenticatie en Helper Functies</b></span>

In deze stap worden de functies en klassen gedefinieerd die nodig zijn om bodemvochtgegevens op te halen, te verwerken en te visualiseren.

##### <span style="color:#C78152"><b>Functionaliteit:</b></span>
* `WIWBAuthClient` klasse:
  * Initialisatie met `client_id` en `client_secret`.
  * Verkrijgt een toegangstoken (`access_token`) van de WIWB API.
  * Controleert of een token verlopen is en vernieuwt indien nodig.
  * Retourneert een HTTP-header met het geldige token voor API-aanvragen.
<br>
* `get_extent(shapefile, waterschap)`:
  * Leest een shapefile in, zet het CRS om naar EPSG:28992 en geeft de grenzen (extent) van het geselecteerde waterschap terug.
<br>
* `build_payload(startdate, enddate, extent)`:
  * Bouwt een JSON payload voor het opvragen van bodemvocht-data via de WIWB API.
<br>
* `fetch_raster(headers, payload, out_path)`:
  * Verstuurd de API-aanvraag en slaat het ontvangen GeoTIFF-bestand lokaal op.
<br>
* `read_raster(file_path, geometry)`:
  * Leest een rasterbestand in en knipt het uit volgens de opgegeven waterschap-geometrie.
<br>
* `create_animation(tif_files, geometry, waterschap_name, start_date, end_date, interval)`:
  * Maakt een animatie van de rasterbestanden voor het geselecteerde waterschap op basis van `start_date` en `end_date`.
<br>

In [ ]:
# -*- coding: utf-8 -*-

# =======================
# AUTHENTICATIE CLASS
# =======================

class WIWBAuthClient:

    def __init__(self, client_id: str, client_secret: str):
        
        # initialize the WIWBClient with the required credentials and API URL
        #
        # Args:
        #    client_id (str): The client ID for authentication
        #    client_secret (str): The client secret for authentication
        #    auth_url (str): The authentication URL of the API
        
        self.client_id = client_id
        self.client_secret = client_secret
        self.auth_url = "https://login.hydronet.com/auth/realms/hydronet/protocol/openid-connect/token"
        self.token = None

    def execute(self):
        
        # step 1: get a valid token, refreshing if necessary
        if self.get_token_expired():
            token = self.get_token()

            if not token:
                raise ValueError("Failed to retrieve a valid token.")
                
            # step 2: create an HTTP header with the valid access token for API requests
            #return token
            return {"content-type": "application/json", "Authorization":
                    "Bearer " + token}
        return None
    
    def get_token_expired(self) -> bool:
        
        # check if the current token has expired
        #
        # Returns:
        #     bool: True if the token is expired, False otherwise
        
        if not self.token:
            return True

        token_decoded = jwt.decode(self.token, options={"verify_signature": False})
        token_exp_datetime = datetime.datetime.fromtimestamp(token_decoded["exp"], tz=datetime.UTC)
        current_datetime = datetime.datetime.now(tz=datetime.UTC) - datetime.timedelta(minutes=1)
        return current_datetime > token_exp_datetime
     
    def get_token(self):

        # fetches an access token from the WIWB API using client credentials
        #
        # Returns:
        #    Optional[str]: the access token if successful, otherwise None
        
        # encode client ID and secret for authentication
        authorization = base64.b64encode(f"{self.client_id}:{self.client_secret}".encode("ISO-8859-1")).decode("ascii")
            
        # create authorization header
        headers = { "Authorization": f"Basic {authorization}", "Content-Type": "application/x-www-form-urlencoded"}
        # create authorization body
        body = {"grant_type": "client_credentials"}

        # make the POST request
        response = requests.post(self.auth_url, data=body, headers=headers)
        response.raise_for_status()  # raise an exception for HTTP errors
        
        # parse and return the access token
        self.token = response.json().get("access_token")
        return self.token


# =======================
# HELPER FUNCTIONS
# =======================

def get_extent(shapefile: Path, waterschap: str) -> Dict:

    # lees shapefile en geef extent terug in EPSG:28992
    #
    # Returns:
    #     xmin, ymin, xmax, ymax: bounds van het waterschap
    
    # selecteer geometrie van het waterschap
    gdf = gpd.read_file(shapefile).to_crs(epsg=28992)
    ws_geom = gdf.loc[gdf["waterschap"] == waterschap, "geometry"]
    if ws_geom.empty:
        raise ValueError(f"Waterschap '{waterschap}' niet gevonden in shapefile.")
    
    # selecteer geometrie van het waterschap
    xmin, ymin, xmax, ymax = ws_geom.values[0].bounds
    return {
        "Xll": xmin,
        "Yll": ymin,
        "Xur": xmax,
        "Yur": ymax,
        "SpatialReference": {"Epsg": 28992}
    }


def build_payload(startdate: str, enddate: str, extent: Dict) -> Dict:
    
    # Bouwt een payload voor het opvragen van satelliet-data (bodemvocht) via de WIWB-dataservice
    #
    # Returns:
    #     dict: Een dictionary die gebruikt kan worden als payload in een API-request

    return {
        "Readers": [
            {
                "DataSourceCode": "LIBV.Bodemvocht.Satelliet.Reanalysis.100m",
                "Settings": {
                    "StructureType": "Grid",
                    "VariableCodes": ["Bodemvocht"],
                    "StartDate": startdate,
                    "EndDate": enddate,
                    "Extent": extent,
                }
            }
        ],
        "Exporter": {
            "DataFormatCode": "geotiff",
            "SpatialReference": {"Epsg": 28992}
        }
    }


def fetch_raster(headers: Dict, payload: Dict, out_path: Path):

    # lees shapefile en geef extent terug in EPSG:28992
    #
    # Returns:
    #     xmin, ymin, xmax, ymax: bounds van het waterschap

    response = requests.post(API_URL, headers=headers, json=payload)
    if response.status_code == 200:
        with open(out_path, "wb") as f:
            f.write(response.content)
        print(f"Opgeslagen: {out_path}")
    else:
        raise RuntimeError(f"Fout {response.status_code}: {response.text}")


def read_raster(file_path, geometry):
    # lees een raster in en knip het uit op de gegeven geometrie
    #
    # Parameters:
    # --------
    #   file_path : Path or str
    #       pad naar het rasterbestand (.tif)
    #   geometry : list
    #       lijst van shapely geometrieën van een waterschap om op te klippen
    # 
    # Returns:
    # --------
    #   np.ndarray : 2D araay van het uitgesneden raster voor het waterschap

    with rasterio.open(file_path) as src:
        out_image, _ = mask(
            src, geometry, crop=True, filled=True, nodata=np.nan 
        )
        arr = out_image[0].astype(float)
        arr[arr == -999] = np.nan
        return arr


def create_animation(tif_files, geometry, waterschap_name, start_date, end_date, interval=500, cmap="viridis", vmin=0, vmax=1, plot_title='Bodemvocht'):
    
    # maak een animatie van de rasterbestanden voor een waterschap
    #
    # Parameters:
    # --------
    #   tif_files : list
    #        lijst van rasterbestanden (.tif)
    #   geometry : list
    #        lijst van shapely geometrieën om op te clippen
    #   waterschap_name : str
    #        naam van het geselecteerde waterschap
    #   interval : int, optional
    #        interval tussen frames in milliseconden (default=500)
    #   cmap : int
    #        type kleurschaal (bijv. "viridis", "plasma", "YlGnBu")
    #   vmin : float
    #        minimum van de kleurschaal (None = automatisch)
    #   vmax : float
    #        maximum van de kleurschaal (None = automatisch)
    #   plot_title : string
    #        titel van de plot
    # 
    # Returns:
    # --------
    #   matplotlib.animation.FuncAnimation : animatie bodemvocht van het waterschap

    # Zorg dat start_date en end_date altijd datetime.date objecten zijn
    if isinstance(start_date, str):
        start_date = datetime.strptime(start_date, "%Y%m%d").date()
    if isinstance(end_date, str):
        end_date = datetime.strptime(end_date, "%Y%m%d").date()
        
    # Filter bestanden op datum indien start_date of end_date opgegeven
    filtered_files = []
    for f in tif_files:
        # haal de datum uit de bestandsnaam (bijvoorbeeld: bodemvocht_libv_historic_20250916.tif)
        date_str = datetime.datetime.strptime(f.stem.split("_")[-1], "%Y%m%d").date()
        if start_date and date_str < start_date:
            continue
        if end_date and date_str > end_date:
            continue
        filtered_files.append(f)

    if len(filtered_files) == 0:
        raise ValueError("Geen bestanden gevonden binnen de opgegeven datumrange.")
    
    # lees de bodemvocht waarden van het eerste figuur
    first_data = read_raster(tif_files[0], geometry)

    # als vmin/vmax None zijn → automatisch bepalen uit eerste raster
    if vmin is None:
        vmin = np.nanmin(first_data)
    if vmax is None:
        vmax = np.nanmax(first_data)


    # intitialiseer de plot
    fig = plt.figure(figsize=(fig_width, fig_height))
    # initialiseer grid voor logo's
    gs = GridSpec(2, 1, height_ratios=[0.10, 1], figure=fig)  
    # bovenste figuur voor de logo's
    ax_logo = fig.add_subplot(gs[0])
    # onderste figuur voor de kaart
    ax = fig.add_subplot(gs[1])

    fig.set_constrained_layout(True)

    # x-as en y-as verbergen van figuur
    ax.axis("off")
    # x-as en y-as verbergen van logo's
    ax_logo.axis("off")


    # inlezen van de logo's
    logo1 = Image.open("static/logo/logo_hydrologic.png")
    logo2 = Image.open("static/logo/logo_planet.png")
    logo3 = Image.open("static/logo/logo_waterschapshuis.png")

    # school (zoom) instellen
    im1 = OffsetImage(logo1, zoom=0.10) 
    im2 = OffsetImage(logo2, zoom=0.05)
    im3 = OffsetImage(logo3, zoom=0.35)

    # plaats ze in de figuur (coördinaten in axes fraction: (0,1) = linkerbovenhoek)
    ax_logo.add_artist(AnnotationBbox(im1, (0.4, 1.5), frameon=False, xycoords="axes fraction"))
    ax_logo.add_artist(AnnotationBbox(im2, (0.65, 1.5), frameon=False, xycoords="axes fraction"))
    ax_logo.add_artist(AnnotationBbox(im3, (0.9, 1.5), frameon=False, xycoords="axes fraction"))


    # intialiseer de plot met de data van het eerste figuur
    im = ax.imshow(first_data, cmap=cmap, vmin=vmin, vmax=vmax)
    # verkrijg de datum van deze file
    date = filtered_files[0].stem.rpartition("_")[-1]
    # initialiseer titel in figuur
    title = ax.set_title(f"{date} - {plot_title}")

    #ax.set_aspect('equal')

    # voel een kleurschaal met waarden toe
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Bodemvocht [m3/m3]")

    def update(frame):
        # update functie dat iteratief de bodemvocht rasters toevoegt
        data = read_raster(filtered_files[frame], geometry)
        im.set_data(data)
        date = filtered_files[frame].stem.rpartition("_")[-1]
        title.set_text(f"{date} - {plot_title}")
        return im, title

    # maak animatie van de bodemvocht rasterbestanden
    ani = FuncAnimation(fig, update, frames=len(filtered_files), interval=interval, blit=False)
    plt.close(fig)
    return ani
    

#### <span style="color:#6495D6"><b>Uitvoering: Ophalen van bodemvocht en genereren animatie</b></span>

In deze stap worden de functies en klassen gedefinieerd die nodig zijn om bodemvochtgegevens op te halen, te verwerken en te visualiseren.

##### <span style="color:#C78152"><b>Stappen van de uitvoering:</b></span>

*  *Authenticatie*: Initialiseer een `WIWBAuthClient` object met `CLIENT_ID` en `CLIENT_SECRET` en verkrijg de HTTP-headers.
*  *Extent ophalen*: Bepaal de grenzen van het geselecteerde waterschap met `get_extent`. Dit wordt gebruikt om het raster te clippen tot het waterschapgebied.
*  *Downloaden van bodemvochtdata*: bouwt payload voor API-aanvraag (`build_payload`) en haalt het GEOTIFF-bestand op met `fetch_raster` voor elke dag in de periode (`start` tot `end`).
*  *Controleer bestanden*: controleer of er `.tif` bestanden aanwezig zijn in de input-map.
*  *Maak animatie*: gebruik `create_animation` om een animatie te maken van alle gevonden GeoTIFFs.

💡 **Tip:**  
Deze cel voert het volledige proces uit van authenticatie, ophalen van bodemvocht raster uit WIWB, tot het maken en tonen van de animatie. Als je een animatie wilt maken van een lange periode kan dit even duren. Bestanden die al bestaan, worden automatisch overgeslagen om tijd te besparen.


In [ ]:
# -*- coding: utf-8 -*-


if __name__ == "__main__":
    
    # initialiseerd WIWBAuthClient object
    wiwbauthclient = WIWBAuthClient(CLIENT_ID, CLIENT_SECRET)
    # execute authenticatie en haal token op
    headers = wiwbauthclient.execute()

    # shapefile extent ophalen
    extent = get_extent(DIR_SHAPE, SELECTED_WATERSCHAP)

    for d in (start + datetime.timedelta(days=i) for i in range((end - start).days + 1)):
        # maak een string van het geselecteerde waterschap
        selected_waterschap = SELECTED_WATERSCHAP.lower().replace(" ", "")
        # initialiseer structuur output bestand
        output_file = DIR_INPUT / f"bodemvocht_libv_{selected_waterschap}_{d.strftime('%Y%m%d')}.tif"

        if output_file.exists():
            # check of output file al bestaat
            print(f"{output_file} bestaat al, overslaan...")
            continue  
        
        # maak de payload voor API call
        payload = build_payload(d.strftime("%Y%m%d") + "000000", d.strftime("%Y%m%d") + "000000", extent)
        # fetch the data
        fetch_raster(headers, payload, output_file)

    # maak string van het geselecteerde waterschap voor filter
    selected_waterschap_str = SELECTED_WATERSCHAP.lower().replace(" ", "")

    # controleer of er .tif bodemvocht bestanden in de folder aanwezig zijn
    tif_files = sorted(DIR_INPUT.glob(f"bodemvocht_libv_{selected_waterschap_str}_*.tif"))
    if len(tif_files) == 0:
        raise FileNotFoundError(f"Geen .tif bestanden gevonden in {DIR_INPUT}")
    print(f"Aantal bestanden gevonden: {len(tif_files)}")

# maak animatie
ani = create_animation(tif_files, geometry, SELECTED_WATERSCHAP, 
                        start_date=start, 
                        end_date=end, 
                        interval=FRAME_INTERVAL,
                            cmap=CMAP,
                            vmin=VMIN,
                            vmax=VMAX,
                            plot_title=PLOT_TITLE)
# Toon animatie in notebook
HTML(ani.to_jshtml())

### Animatie opslaan

In [ ]:
from matplotlib.animation import FFMpegWriter

output_file = DIR_OUTPUT / (
    "bodemvocht_%s_%s_%s.mp4" % (
        SELECTED_WATERSCHAP.replace(" ", "_"),
        start.strftime("%Y%m%d"),
        end.strftime("%Y%m%d"))
)

writer = FFMpegWriter(fps=2, bitrate=1800)
ani.save(output_file, writer=writer)
print(f"Animatie opgeslagen als: {output_file}")